# LangChain Basics
**Topics covered:** what LangChain is and why · setup · chat models · prompt templates · output parsers · chains with LCEL · streaming and batch

> Last notebooks: you built RAG by hand (chunk, embed, store, retrieve, prompt, LLM). LangChain wraps those same pieces so you write less glue code. This notebook covers the building blocks; the next one rebuilds RAG with them.

---
## PART 1: What Is LangChain?

**LangChain is a framework that gives standard building blocks for LLM apps**, so you don't rewrite the same glue code each time.

By hand, every LLM app repeats the same chores: format a prompt string, call a provider's API, dig the text out of the response, pass it to the next step. LangChain turns each chore into a small reusable piece, and lets you snap the pieces together.

**Analogy:** LEGO bricks. You could carve every piece from wood yourself (what you did in the RAG notebooks), or use standard bricks that click together. The bricks don't make the model smarter; they make building and changing the app faster.

### The four pieces in this notebook

| Piece | What it is | Input -> Output | Analogy |
|---|---|---|---|
| **Chat model** | A wrapper around the LLM's API. It sends messages and gets a reply. | messages -> `AIMessage` (reply) | The expert you ask the question |
| **Prompt template** | A prompt with `{blanks}` that you fill in at run time. | values (like `topic`) -> messages | A form letter: fixed wording, changing details |
| **Output parser** | Converts the model's reply into the form your code needs (string, dict, etc.). | `AIMessage` -> `str` or `dict` | A translator that turns the reply into something usable |
| **Chain** | The pieces connected in order, so each output feeds the next. | input -> final result | An assembly line: each station does one job and hands the result on |


```
input -> prompt template -> chat model -> output parser -> result
```

---
## PART 2: Setup

You do this yourself, once:

1. Install: `pip install langchain-groq` (this also installs `langchain-core`, which holds the prompt, parser and chain pieces)
2. The key stays in your `.env` as `GROQ_API_KEY=...`, same as before

If no key is found, the notebook still runs. Cells that need the LLM are skipped with a message, so the rest keeps working.

> **Model names change.** `MODEL` is set in one place below. If you get a "model not found" error, list the available models and paste a current chat model name here.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()                                                 # reads GROQ_API_KEY from .env
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"                                 # check console.groq.com for the current model list

if HAS_KEY:
    from langchain_groq import ChatGroq
    llm = ChatGroq(model=MODEL, temperature=0)                # temperature 0 = most predictable answers

print("LLM available:", HAS_KEY)

LLM available: True


---
## PART 3: Chat Models

A **chat model** object wraps the provider's API. Every LangChain model has the same methods, so switching from Groq to another provider later means changing one line, not rewriting your app.

The main method is **`.invoke(input)`**: send something in, get one reply back.
A chat model is an AI you send messages to, and it sends a message back.

In LangChain it's one Python object, like ChatGroq, that handles the connection to the AI for you. You call it with llm.invoke("your question") and get a reply.

Think of it as texting an expert. You write a message, and they write one back.

In [4]:
if HAS_KEY:
    reply = llm.invoke("In one sentence, what is a vector database?")
    print(type(reply).__name__)
    print(reply.content)
else:
    print("[dry run: no GROQ_API_KEY found, so no LLM was called]")

AIMessage
A vector database is a specialized data store that indexes and retrieves high‑dimensional vector embeddings—often generated by machine‑learning models—to enable fast similarity search and nearest‑neighbor queries across large collections of unstructured data.


### What came back?

`reply` is not a plain string. It is an **`AIMessage`** object: the text is in `.content`, and extra info (token counts, model name) sits next to it.

### Messages and roles

Chat models take a **list of messages**, each with a role:

| Message | Who writes it | What it does | Example |
|---|---|---|---|
| `SystemMessage` | You, the developer | Sets the rules and behavior for the whole chat. It comes first and stays in place. | "You are a patient teacher. Answer in two sentences." |
| `HumanMessage` | The user | The question or request from the person. | "What is an embedding?" |
| `AIMessage` | The model | The model's reply. You also add past replies back in, so the model can see the conversation so far. | "An embedding turns text into numbers that capture meaning." |



In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("You are a patient teacher. Answer in at most two sentences."),
    HumanMessage("What is an embedding?"),
]

if HAS_KEY:
    reply = llm.invoke(messages)
    print(reply.content)
    print("\nmetadata:", reply.response_metadata.get("token_usage"))
else:
    print("[dry run: no GROQ_API_KEY found, so no LLM was called]")

An embedding is a way of representing items—such as words, images, or nodes—in a continuous vector space so that similar items end up with similar coordinates. This compact numeric form lets machine‑learning models process and compare complex data efficiently.

metadata: {'completion_tokens': 76, 'prompt_tokens': 92, 'total_tokens': 168, 'completion_time': 0.158612803, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.004421056, 'prompt_tokens_details': None, 'queue_time': 0.212626505, 'total_time': 0.163033859}


---
## PART 4: Prompt Templates

**Prompt template: a prompt with blanks**

- A prompt template is a prompt written once, with `{blanks}` that you fill in at run time.
- It's the same idea as your `build_prompt()` function from the RAG notebook, packaged as a reusable object.
- **Analogy:** a form letter. The wording stays fixed, and only the name and date change.

**Example**
```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain AI concepts to a beginner."),
    ("human", "Explain {topic} in {n_sentences} sentences."),
])
prompt.invoke({"topic": "chunking", "n_sentences": 2})


In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain AI concepts to a beginner. Be short and use one analogy."),
    ("human", "Explain {topic} in {n_sentences} sentences."),
])

print("blanks it needs:", prompt.input_variables)

filled = prompt.invoke({"topic": "chunking", "n_sentences": 2})
for m in filled.messages:
    print(f"{type(m).__name__}: {m.content}")

blanks it needs: ['n_sentences', 'topic']
SystemMessage: You explain AI concepts to a beginner. Be short and use one analogy.
HumanMessage: Explain chunking in 2 sentences.


No LLM was called above. A template only builds the messages. That is useful for debugging: you can **print exactly what will be sent** before spending a call (the same habit as "print the full prompt" in the RAG debugging order).

**Why use a template instead of an f-string?**
- It checks that every blank is filled, so a missing value gives a clear error.
- It produces proper system and human messages, not one long string.
- It plugs straight into a chain (Part 6).

---
## PART 5: Output Parsers

The model returns an `AIMessage`, but your code usually wants a plain string, or structured data. An **output parser** converts the reply into the form you need.

| Parser | Gives you | Example |
|---|---|---|
| `StrOutputParser` | A plain string (just the text) | `AIMessage("Hi")` -> `"Hi"` |
| `JsonOutputParser` | A Python dict, parsed from JSON in the reply | `'{"term": "chunking"}'` -> `{"term": "chunking"}` |

**Things to remember**
- A parser works on a reply you already have, so you can test it without calling the LLM.
- `JsonOutputParser` only **parses**. It does not make the model write JSON. You have to ask for JSON in the prompt.
- If the model's reply isn't valid JSON, the parser raises an error.
- With `StrOutputParser` at the end of a chain, you get a string directly and don't need `.content`.

In [7]:
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

fake_reply = AIMessage(content="Chunking splits a document into small pieces.")
print(repr(StrOutputParser().invoke(fake_reply)))             # AIMessage -> plain str

fake_json_reply = AIMessage(content='{"term": "chunking", "difficulty": "easy"}')
print(JsonOutputParser().invoke(fake_json_reply))             # JSON text -> dict

'Chunking splits a document into small pieces.'
{'term': 'chunking', 'difficulty': 'easy'}


---
## PART 6: Chains with LCEL

A **chain** connects the pieces so the output of one feeds the next. LangChain's way to write this is **LCEL** (LangChain Expression Language), using the pipe symbol `|`:

```python
chain = prompt | llm | parser
```

Read it left to right, like a Unix pipe: fill the prompt, send it to the model, parse the reply.

| Step | Job | Input → Output |
|---|---|---|
| `prompt` | Fills the template with your variables | `dict` → messages |
| `llm` | Sends the messages to the model | messages → `AIMessage` |
| `parser` | Cleans up the reply | `AIMessage` → `str` / `dict` / object |

**Analogy:** an assembly line. Each station does one job and passes the result along.

A chain is itself a runnable object, with the same `.invoke()` method as the model. That is why chains can be nested inside bigger chains.

### Why use a chain?

- **Less glue code:** you don't call each step by hand.
- **Reusable:** swap the model or prompt without rewriting the flow.
- **Batch, stream and async come for free** on every chain.
- **Easy to compose:** small chains combine into larger ones.

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You explain AI concepts to a beginner. Be short and use one analogy."),
    ("human", "Explain {topic} in {n_sentences} sentences."),
])

if HAS_KEY:
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"topic": "reranking", "n_sentences": 2})
    print(answer)
else:
    print("[dry run: no GROQ_API_KEY found, so no LLM was called]")

Reranking is like a teacher first giving a class a quick multiple‑choice quiz (the initial model) and then reviewing the top answers more carefully to pick the best one (the second model). It takes the shortlist of candidates and re‑orders them using a more precise, often larger, model to improve the final result.


The result is already a plain string, because `StrOutputParser` is the last step. Compare with Part 3, where you got an `AIMessage` and had to read `.content` yourself.

### A chain that returns structured data

Same pattern, with `JsonOutputParser` at the end and format instructions in the prompt.

In [9]:
json_parser = JsonOutputParser()

json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reply only with JSON. {format_instructions}"),
    ("human", "Give a one-line definition and a difficulty (easy, medium or hard) for the term: {term}"),
]).partial(format_instructions=json_parser.get_format_instructions())

if HAS_KEY:
    json_chain = json_prompt | llm | json_parser
    result = json_chain.invoke({"term": "cosine similarity"})
    print(type(result).__name__, result)
else:
    print("[dry run: no GROQ_API_KEY found, so no LLM was called]")

dict {'definition': 'Cosine similarity measures the cosine of the angle between two non‑zero vectors, indicating how similar their directions are.', 'difficulty': 'easy'}


`.partial(...)` fills some blanks ahead of time, so the chain only needs `term` at run time.

If the model wraps the JSON in extra text, the parser raises an error. Models that follow instructions well rarely do this.

---
## PART 7: Streaming and Batch

Every chain is a Runnable, so it gets these extra ways to run for free, with no new code:

| Method | Input | Returns | Use it for |
|---|---|---|---|
| `.invoke(x)` | One input | One full result, after the model finishes | Normal calls |
| `.stream(x)` | One input | A generator of chunks, yielded as they are generated | Chat UIs that show text as it types |
| `.batch([x, y, z])` | A list of inputs | A list of results, in the same order | Processing many questions at once |

Async versions: `.ainvoke()`, `.astream()`, `.abatch()`. Use these inside async apps such as FastAPI.

**Analogy:** `.invoke` is waiting for the whole letter to arrive. `.stream` is reading it as it is being typed. `.batch` is sending ten letters at the same time.

**Remember:**
- `.batch` runs inputs in parallel, so it is much faster than a `for` loop of `.invoke` calls. Limit concurrency with `config={"max_concurrency": 5}` to avoid rate limits.
- `.stream` only streams if every step after the model can handle chunks. `StrOutputParser` can, so `prompt | llm | StrOutputParser()` streams fine.

In [10]:
if HAS_KEY:
    chain = prompt | llm | StrOutputParser()

    print("STREAM:")
    for piece in chain.stream({"topic": "an embedding", "n_sentences": 2}):
        print(piece, end="", flush=True)

    print("\n\nBATCH:")
    topics = ["chunking", "top-k", "hybrid search"]
    answers = chain.batch([{"topic": t, "n_sentences": 1} for t in topics])
    for t, a in zip(topics, answers):
        print(f"- {t}: {a}")
else:
    print("[dry run: no GROQ_API_KEY found, so no LLM was called]")

STREAM:
An embedding is a way to turn items—like words, images, or users—into a list of numbers that captures their meaning or relationships. Think of it like placing each book on a library shelf so that similar books end up close together; the numbers are the coordinates that tell you where each “book” belongs.

BATCH:
- chunking: Chunking is like packing a suitcase by folding related items into bundles, letting you remember or handle many pieces of information as a single, manageable “package.”
- top-k: Top‑k is like picking the k tallest people from a crowd— you keep only the highest‑scoring items (the “tallest”) and ignore the rest.
- hybrid search: Hybrid search is like using both a map and a compass together—it combines fast keyword matching (the map) with deep semantic understanding (the compass) to pinpoint the most relevant results.


**Why streaming matters:** an LLM can take several seconds to finish a long answer. Streaming shows the first words in under a second, so the app feels fast even though the total time is the same.

---
## PART 8: Summary

| Piece | What it is | Key call |
|---|---|---|
| **Chat model** | Wrapper over the provider API | `ChatGroq(model=...)`, `.invoke(...)` |
| **Messages** | System / Human / AI roles | `SystemMessage`, `HumanMessage` |
| **Prompt template** | Prompt with `{blanks}` | `ChatPromptTemplate.from_messages(...)` |
| **Output parser** | Reply into str or dict | `StrOutputParser`, `JsonOutputParser` |
| **Chain (LCEL)** | Pieces joined with `\|` | `prompt \| llm \| parser` |
| **Run modes** | Same chain, three ways | `.invoke`, `.stream`, `.batch` |

### Key ideas

- LangChain gives **standard, swappable pieces**; it doesn't add intelligence to the model.
- A chain is `prompt | model | parser`, read left to right.
- A template can be **printed before you call the model**, which makes debugging easy.
- `.stream` and `.batch` come free on every chain.

### Check yourself

**1. What does the `|` in `prompt | llm | parser` do?**
- It connects the pieces into a chain, so the output of one becomes the input of the next.
- The prompt fills its blanks and builds messages. The model returns an `AIMessage`. The parser turns that into a string or dict.

**2. Why does `reply.content` exist, and what parser removes the need for it?**
- The model returns an `AIMessage` object, not a plain string. It holds the text plus extras like token counts and metadata.
- `.content` pulls out just the text.
- `StrOutputParser` at the end of a chain does this for you, so you get a plain string directly.

**3. Does `JsonOutputParser` make the model write JSON? If not, what does?**
- No. It only **parses** JSON text that is already in the reply, turning it into a dict.
- Your **prompt** is what makes the model write JSON. Ask for it, for example by pasting in `parser.get_format_instructions()`.
- If the reply isn't valid JSON, the parser raises an error.

**4. Which chain method would you use for a chat window that shows text as it appears?**
- `.stream()`. It gives the reply piece by piece as it is generated, so the first words show up quickly.
- `.invoke()` makes the user wait for the whole reply.

